In [1]:
%reload_ext jupyter_ai

In [2]:
%env FREVAGPT_API_KEY=***REMOVED***

env: FREVAGPT_API_KEY=***REMOVED***


In [13]:
%%ai FrevaGPT-provider:test-model-1 -f code
Write code to calculate the average temperature for a given year range and a given lat lon box.

test-model-1


In [ ]:
import xarray as xr

def calculate_average_temperature(data_path, start_year, end_year, lat_min, lat_max, lon_min, lon_max):
    # Load the dataset
    dset = xr.open_mfdataset(data_path)

    # Select the time range
    time_slice = slice(f'{start_year}-01-01', f'{end_year}-12-31')
    dset_time_filtered = dset.sel(time=time_slice)

    # Select the spatial box
    dset_spatial_filtered = dset_time_filtered.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))

    # Calculate the mean temperature over the specified time and space
    average_temperature = dset_spatial_filtered['tas'].mean(dim=['time', 'lat', 'lon'])

    return average_temperature.values

In [14]:
%%ai FrevaGPT-provider:test-model-1 
Please explain the code that you just created.

test-model-1


```markdown
The provided Python code is a function that calculates the average temperature over a specified year range and geographical box defined by latitude and longitude boundaries. Here's a step-by-step explanation:

1. **Import the Required Library**:
   - The code imports the `xarray` library, which is used for handling multi-dimensional arrays such as those found in climate data.

2. **Define the Function**:
   - The function `calculate_average_temperature` takes the following parameters:
     - `data_path`: The path to the dataset files.
     - `start_year` and `end_year`: The range of years for which the average temperature is to be calculated.
     - `lat_min`, `lat_max`, `lon_min`, `lon_max`: The latitude and longitude boundaries defining the geographical box.

3. **Load the Dataset**:
   - The dataset is loaded using `xr.open_mfdataset(data_path)`, which can handle multiple files and combines them into a single dataset.

4. **Select the Time Range**:
   - A time slice is created using the specified start and end years.
   - The dataset is filtered to include only data within this time range using `dset.sel(time=time_slice)`.

5. **Select the Spatial Box**:
   - The dataset is further filtered to include only data within the specified latitude and longitude boundaries using `dset.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))`.

6. **Calculate the Mean Temperature**:
   - The mean temperature is calculated over the specified time and space dimensions using `dset_spatial_filtered['tas'].mean(dim=['time', 'lat', 'lon'])`.
   - This computes the average temperature across all time steps and geographical points within the selected range.

7. **Return the Result**:
   - The function returns the computed average temperature as a scalar value using `average_temperature.values`.

This code allows users to easily compute average temperatures for specific regions and time periods from climate datasets. 
```

In [22]:
%ai reset

In [3]:
%%ai FrevaGPT-provider:test-model-1 -f code 
Can you now create some code with which to calculate and plot the climatology of tas for Germany between the years 1980 and 2010? 

test-model-1


In [ ]:
import freva
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Access the temperature data for the years 1980 to 2010 using the databrowser
files = freva.databrowser(project='reanalysis', experiment='era5', variable='tas', time_frequency='mon', time='1980-01-01to2010-12-31')

# Load the data using xarray
dset = xr.open_mfdataset(files, combine='by_coords')

# Select the data for Germany (approximate lat/lon bounds)
germany_tas = dset['tas'].sel(lat=slice(47.27, 55.06), lon=slice(5.87, 15.04))

# Calculate the climatology (mean over the specified period)
climatology_tas = germany_tas.mean(dim='time')

# Plot the climatology
plt.figure(figsize=(8, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([5.87, 15.04, 47.27, 55.06], crs=ccrs.PlateCarree())
contour = ax.contourf(dset['lon'], dset['lat'], climatology_tas, cmap='coolwarm', extend='both')
plt.colorbar(contour, label='Temperature (K)')
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS)
plt.title('Climatology of Temperature (1980-2010) for Germany')
plt.show()

In [4]:
%%ai FrevaGPT-provider:test-model-1 -f markdown
Please explain the code that you just created, line by line.

test-model-1


```markdown
1. **Import Libraries**:  
   ```python
   import freva
   import xarray as xr
   import matplotlib.pyplot as plt
   import cartopy.crs as ccrs
   import cartopy.feature as cfeature
   ```
   - Import necessary libraries for data handling (`freva`, `xarray`) and visualization (`matplotlib`, `cartopy`).

2. **Access Temperature Data**:  
   ```python
   files = freva.databrowser(project='reanalysis', experiment='era5', variable='tas', time_frequency='mon', time='1980-01-01to2010-12-31')
   ```
   - Use the `freva.databrowser` to access monthly temperature data (`tas`) from the ERA5 reanalysis project for the period 1980 to 2010.

3. **Load Data with xarray**:  
   ```python
   dset = xr.open_mfdataset(files, combine='by_coords')
   ```
   - Load the accessed data files into an `xarray.Dataset`, combining them by coordinates.

4. **Select Data for Germany**:  
   ```python
   germany_tas = dset['tas'].sel(lat=slice(47.27, 55.06), lon=slice(5.87, 15.04))
   ```
   - Select temperature data for Germany using approximate latitude and longitude bounds.

5. **Calculate Climatology**:  
   ```python
   climatology_tas = germany_tas.mean(dim='time')
   ```
   - Compute the climatology by calculating the mean temperature over the specified time period (1980-2010).

6. **Plot Climatology**:  
   ```python
   plt.figure(figsize=(8, 6))
   ax = plt.axes(projection=ccrs.PlateCarree())
   ax.set_extent([5.87, 15.04, 47.27, 55.06], crs=ccrs.PlateCarree())
   contour = ax.contourf(dset['lon'], dset['lat'], climatology_tas, cmap='coolwarm', extend='both')
   plt.colorbar(contour, label='Temperature (K)')
   ax.add_feature(cfeature.COASTLINE)
   ax.add_feature(cfeature.BORDERS)
   plt.title('Climatology of Temperature (1980-2010) for Germany')
   plt.show()
   ```
   - Create a plot of the climatology:
     - Set up a figure and axis with a Plate Carree projection.
     - Define the extent to focus on Germany.
     - Use `contourf` to plot filled contours of the climatology data.
     - Add a colorbar to indicate temperature values in Kelvin.
     - Add coastlines and borders for geographical context.
     - Set the plot title and display the plot.
```